In [80]:
from datetime import date

import pandas as pd
import yaml
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

CONEXION A BASE DE DATOS

In [81]:

#ABRIR ARCHIVO DE CONFIGURACION DE CONEXION A BASES DE DATOS
with open('../configuracion.yml', 'r') as f:
    configuracion = yaml.safe_load(f)
    configuracionBaseDatos= configuracion['ADVENTURE_WORKS_DB']
    configuracionBodegaDatos= configuracion['ADVENTURE_WORKS_DW']

# CREAR LAS URLs DE CONEXION

urlBaseDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBaseDatos['user'],
    password=str(configuracionBaseDatos['password']),
    host=configuracionBaseDatos['host'],
    port=configuracionBaseDatos['port'],
    database=configuracionBaseDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)

urlBodegaDatos = URL.create(
    "mssql+pyodbc",
    username=configuracionBodegaDatos['user'],
    password=str(configuracionBodegaDatos['password']),
    host=configuracionBodegaDatos['host'],
    port=configuracionBodegaDatos['port'],
    database=configuracionBodegaDatos['dbname'],
    query={"driver": "ODBC Driver 17 for SQL Server"}
)



# CREAR EL MOTOR DE SQLALCHEMY
motorBaseDatos = create_engine(urlBaseDatos)
motorBodegaDatos = create_engine(urlBodegaDatos)

EXTRACCION

In [ ]:

queryPurchaseOderDetail = """
SELECT 
[PurchaseOrderID]
      ,[PurchaseOrderDetailID]
      ,[DueDate]
      ,[OrderQty]
      ,[ProductID]
      ,[UnitPrice]
      ,[LineTotal]
FROM Purchasing.PurchaseOrderDetail
"""

tablaPurchaseOderDetail = pd.read_sql_query(queryPurchaseOderDetail, motorBaseDatos)




queryPurchaseOderHeader = """
SELECT 
[PurchaseOrderID]
      ,[RevisionNumber]
      ,[EmployeeID]
      ,[OrderDate]
      ,[ShipDate]
      ,[TaxAmt]
      ,[Freight]
FROM Purchasing.PurchaseOrderHeader
"""

tablaPurchaseOderHeader = pd.read_sql_query(queryPurchaseOderHeader, motorBaseDatos)



queryProduct = """
SELECT 
[ProductKey]
      ,[StandardCost]
FROM dbo.dimensionProduct
"""

dimensionProduct = pd.read_sql_query(queryProduct, motorBodegaDatos)



queryPromotion = """
SELECT 
[PromotionKey]
      ,[DiscountPct]
FROM dbo.dimensionPromotion
"""

dimensionPromotion = pd.read_sql_query(queryPromotion, motorBodegaDatos)



querySpecialOfferProduct = """
SELECT 
[SpecialOfferID]
      ,[ProductID]
FROM Sales.SpecialOfferProduct
"""

tablaSpecialOfferProduct = pd.read_sql_query(querySpecialOfferProduct, motorBaseDatos)


# tablaPurchaseOderDetail
# tablaPurchaseOderHeader
# dimensionProduct
# dimensionPromotion
# tablaSpecialOfferProduct

TRANSFORMACION

In [83]:
promotion = dimensionPromotion.merge(tablaSpecialOfferProduct, left_on='PromotionKey', right_on='SpecialOfferID')
promotion = promotion.merge(dimensionProduct,  left_on='ProductID', right_on='ProductKey')

promotion.drop(columns=[
    'SpecialOfferID',
    'ProductKey'
], inplace=True)


promotion

,PromotionKey,DiscountPct,ProductID,StandardCost
0,1,0.0,680,1059.3100
1,1,0.0,680,1059.3100
2,1,0.0,680,1059.3100
3,1,0.0,680,1059.3100
4,1,0.0,680,1059.3100
...,...,...,...,...
3217,16,0.4,988,308.2179
3218,16,0.4,988,308.2179
3219,16,0.4,988,308.2179
3220,16,0.4,988,308.2179


In [84]:
tablaPurchaseSales = tablaPurchaseOderDetail.merge(tablaPurchaseOderHeader, on='PurchaseOrderID')
tablaPurchaseSales = tablaPurchaseSales.merge(promotion, on='ProductID')
tablaPurchaseSales

,PurchaseOrderID,PurchaseOrderDetailID,DueDate,OrderQty,ProductID,UnitPrice,LineTotal,RevisionNumber,EmployeeID,OrderDate,ShipDate,TaxAmt,Freight,PromotionKey,DiscountPct,StandardCost
0,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,866.1056,1,0.0,35.9596
1,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,866.1056,1,0.0,35.9596
2,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,866.1056,1,0.0,35.9596
3,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,866.1056,1,0.0,35.9596
4,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,866.1056,1,0.0,35.9596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18349,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,19953.6000,4,0.1,41.5723
18350,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,19953.6000,4,0.1,41.5723
18351,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,19953.6000,4,0.1,41.5723
18352,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,19953.6000,4,0.1,41.5723


In [85]:

tablaPurchaseSales.rename(columns={
    'StandardCost': 'ProductStandardCost',
    'TerritoryID': 'SalesTerritoryKey',
    'CustomerID': 'CustomerKey',
    'UnitPriceDiscount': 'UnitPriceDiscountPct',
    'SpecialOfferID': 'PromotionKey',
    'OrderQty': 'OrderQuantity',
    'LineTotal' : 'ExtendedAmount',
    'EmployeeID' : 'EmployeeKey',
    'ProductID' : 'ProductKey',
    'DiscountPct' : 'UnitPriceDiscountPct'
}, inplace=True)

tablaPurchaseSales["OrderDateKey"] = pd.to_datetime(tablaPurchaseSales["OrderDate"]).dt.strftime('%Y%m%d')
tablaPurchaseSales["DueDateKey"] = pd.to_datetime(tablaPurchaseSales["DueDate"]).dt.strftime('%Y%m%d')
tablaPurchaseSales["ShipDateKey"] = pd.to_datetime(tablaPurchaseSales["ShipDate"]).dt.strftime('%Y%m%d')


tablaPurchaseSales["SalesAmount"] = tablaPurchaseSales["ExtendedAmount"] 
tablaPurchaseSales["CustomerPONumber"] = None
tablaPurchaseSales["ResellerKey"] = None
tablaPurchaseSales["CurrencyKey"] = None
tablaPurchaseSales["CarrierTrackingNumber"] = None
tablaPurchaseSales["CustomerPONumber"] = None
tablaPurchaseSales["SalesOrderLineNumber"] = None
tablaPurchaseSales["SalesOrderNumber"] = None
tablaPurchaseSales["SalesTerritoryKey"] = None


tablaPurchaseSales

,PurchaseOrderID,PurchaseOrderDetailID,DueDate,OrderQuantity,ProductKey,UnitPrice,ExtendedAmount,RevisionNumber,EmployeeKey,OrderDate,...,DueDateKey,ShipDateKey,SalesAmount,CustomerPONumber,ResellerKey,CurrencyKey,CarrierTrackingNumber,SalesOrderLineNumber,SalesOrderNumber,SalesTerritoryKey
0,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,20111228,20111223,34644.225,None,None,None,None,None,None,None
1,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,20111228,20111223,34644.225,None,None,None,None,None,None,None
2,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,20111228,20111223,34644.225,None,None,None,None,None,None,None
3,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,20111228,20111223,34644.225,None,None,None,None,None,None,None
4,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,20111228,20111223,34644.225,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18349,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,20140724,20140719,249420.000,None,None,None,None,None,None,None
18350,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,20140724,20140719,249420.000,None,None,None,None,None,None,None
18351,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,20140724,20140719,249420.000,None,None,None,None,None,None,None
18352,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,20140724,20140719,249420.000,None,None,None,None,None,None,None


In [86]:
tablaPurchaseSales["DiscountAmount"] = tablaPurchaseSales["UnitPrice"] *  tablaPurchaseSales["UnitPriceDiscountPct"] * tablaPurchaseSales["OrderQuantity"]
tablaPurchaseSales["TotalProductCost"] = tablaPurchaseSales["ProductStandardCost"] *  tablaPurchaseSales["OrderQuantity"]
tablaPurchaseSales

,PurchaseOrderID,PurchaseOrderDetailID,DueDate,OrderQuantity,ProductKey,UnitPrice,ExtendedAmount,RevisionNumber,EmployeeKey,OrderDate,...,SalesAmount,CustomerPONumber,ResellerKey,CurrencyKey,CarrierTrackingNumber,SalesOrderLineNumber,SalesOrderNumber,SalesTerritoryKey,DiscountAmount,TotalProductCost
0,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
1,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
2,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
3,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
4,12,28,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18349,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80
18350,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80
18351,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80
18352,4012,8845,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80


In [87]:
tablaPurchaseSales.drop(columns=[
    'PurchaseOrderDetailID',
    'PurchaseOrderID'
], inplace=True)
tablaPurchaseSales

,DueDate,OrderQuantity,ProductKey,UnitPrice,ExtendedAmount,RevisionNumber,EmployeeKey,OrderDate,ShipDate,TaxAmt,...,SalesAmount,CustomerPONumber,ResellerKey,CurrencyKey,CarrierTrackingNumber,SalesOrderLineNumber,SalesOrderNumber,SalesTerritoryKey,DiscountAmount,TotalProductCost
0,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
1,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
2,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
3,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
4,2011-12-28,550,941,62.9895,34644.225,4,254,2011-12-14,2011-12-23,2771.538,...,34644.225,None,None,None,None,None,None,None,0.0,19777.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18349,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80
18350,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80
18351,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80
18352,2014-07-24,6000,884,41.5700,249420.000,9,254,2014-06-24,2014-07-19,79814.400,...,249420.000,None,None,None,None,None,None,None,24942.0,249433.80


In [88]:
tablaPurchaseSales.columns


Index(['DueDate', 'OrderQuantity', 'ProductKey', 'UnitPrice', 'ExtendedAmount',
       'RevisionNumber', 'EmployeeKey', 'OrderDate', 'ShipDate', 'TaxAmt',
       'Freight', 'PromotionKey', 'UnitPriceDiscountPct',
       'ProductStandardCost', 'OrderDateKey', 'DueDateKey', 'ShipDateKey',
       'SalesAmount', 'CustomerPONumber', 'ResellerKey', 'CurrencyKey',
       'CarrierTrackingNumber', 'SalesOrderLineNumber', 'SalesOrderNumber',
       'SalesTerritoryKey', 'DiscountAmount', 'TotalProductCost'],
      dtype='object')

CARGAR A LA BODEGA

In [90]:
tablaPurchaseSales.to_sql('hechoResellerSales',motorBodegaDatos, if_exists='replace',index=False)

28